## Actividad 3_20: Perros y gatos
<div style="border-style:groove;border-width:thin;padding:10px">
En esta actividad vamos a utilizar las técnicas de redes neuronales y deep learning que hemos visto en clase para enseñar a este software a diferenciar entre perros y gatos.

Para ello vamos a cargar los datos y etiquetarlos, a lanzar un Random Forest Classifier para establecer un punto de partida que debemos mejorar y después, vamos a tratar de solucionar el problema con una red neuronal convencional.
</div>

In [204]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow import keras
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

photos =  []
labels = []

In [205]:
folders = listdir('PetImages')

for idx, folder in enumerate(folders):

    for file in listdir('PetImages/' + folder )[:10000]:
        photo = load_img('PetImages/'+folder+'/'+file, target_size=(128,128))
        photo = img_to_array(photo) 
        photos.append(photo)
        labels.append(float(idx))
  
    print(idx) 

0


/home/ciabd12/anaconda3/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


1


In [206]:
print(len(photos))

20000


In [207]:
photos = asarray(photos, dtype='float32') / 255.0
# photos = photos.reshape(len(photos), -1) / 255
labels = asarray(labels)

In [208]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(photos, labels, test_size=0.2, random_state=42)

In [47]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create and train the Random Forest Classifier
rf_classifier = RandomForestClassifier(min_samples_split=5,n_estimators=100,random_state=42)
rf_classifier.fit(X_train, y_train)

# Make predictions on test set
y_pred = rf_classifier.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest Classifier Accuracy: {accuracy:.4f}")

Random Forest Classifier Accuracy: 0.6350


In [48]:
X_train.shape[1]

49152

In [ ]:
#entrenar una red neuronal con los datos transformados por PCA

model = keras.models.Sequential()
model.add(keras.layers.Dense(1024, activation='relu',kernel_initializer='he_normal', input_shape=(X_train.shape[1],)))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(512, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1, activation='sigmoid', kernel_initializer='glorot_normal'))

In [209]:
#entrenar una red neuronal con los datos transformados por PCA

model = keras.models.Sequential()
model.add(keras.layers.Conv2D(16, (3,3),activation='relu',input_shape=(X_train.shape[1:])))
model.add(keras.layers.MaxPool2D((2,2)))
model.add(keras.layers.Conv2D(32, (3,3),activation='relu'))
model.add(keras.layers.MaxPool2D((2,2)))
model.add(keras.layers.Conv2D(64, (3,3),activation='relu'))
model.add(keras.layers.MaxPool2D((2,2)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(200, activation='relu',kernel_initializer='he_normal'))
# model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(40, activation='relu', kernel_initializer='he_normal'))
# model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1, activation='sigmoid', kernel_initializer='glorot_normal'))

In [210]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.00001, beta_1=0.9, beta_2=0.999), loss='binary_crossentropy', metrics=['accuracy'])

In [211]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100000,validation_split = 0.1,callbacks=[early_stopping_cb])

Epoch 1/100000
450/450 ━━━━━━━━━━━━━━━━━━━━ 20s 43ms/step - accuracy: 0.5722 - loss: 0.6783 - val_accuracy: 0.6162 - val_loss: 0.6655
Epoch 2/100000
450/450 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.6394 - loss: 0.6439 - val_accuracy: 0.6388 - val_loss: 0.6428
Epoch 3/100000
450/450 ━━━━━━━━━━━━━━━━━━━━ 19s 42ms/step - accuracy: 0.6656 - loss: 0.6180 - val_accuracy: 0.6556 - val_loss: 0.6204
Epoch 4/100000
450/450 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.6847 - loss: 0.5966 - val_accuracy: 0.6775 - val_loss: 0.6067
Epoch 5/100000
450/450 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - accuracy: 0.6983 - loss: 0.5821 - val_accuracy: 0.6762 - val_loss: 0.6017
Epoch 6/100000
450/450 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - accuracy: 0.7070 - loss: 0.5697 - val_accuracy: 0.7025 - val_loss: 0.5840
Epoch 7/100000
450/450 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.7182 - loss: 0.5561 - val_accuracy: 0.7131 - val_loss: 0.5718
Epoch 8/100000
450/450 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - ac

In [212]:
import numpy as np
# y_test = np.array(y_test, dtype='int32')
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Loss: {loss}, Accuracy: {accuracy}')

125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7922 - loss: 0.4656
Loss: 0.4656377136707306, Accuracy: 0.7922499775886536
